Pydantic 
is a data validation and settings management library for Python, based on Python type annotations. It allows you to define data models with type hints, and it automatically validates the data against those types. Pydantic is commonly used in web development frameworks like FastAPI to ensure that incoming request data is valid and conforms to the expected schema.


In [17]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [18]:
model = init_chat_model("openai/gpt-oss-20b", model_provider="groq")

In [29]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the moview was released")
    director:str=Field(description="The director of the movie")
    rating:str=Field(description="The movies rating out of 10")
    

In [32]:
model_so = model.with_structured_output(Movie,include_raw=True)

In [33]:
model_so.invoke("provides dettails about the movie Inception")

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "User wants details about the movie Inception. We have a function to call: functions.Movie with director, rating, title, year. We need to provide details. We'll call the function.", 'tool_calls': [{'id': 'fc_ac66061e-2068-47aa-b8a0-67c991804819', 'function': {'arguments': '{"director":"Christopher Nolan","rating":"8.8","title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 161, 'total_tokens': 239, 'completion_time': 0.081083944, 'completion_tokens_details': {'reasoning_tokens': 39}, 'prompt_time': 0.007836329, 'prompt_tokens_details': None, 'queue_time': 0.346695576, 'total_time': 0.088920273}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_996f667773', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0faf3-7b23-7253-a69b-9000693f948a-

Nested Structures in Pydantic allow you to define complex data models that can contain other models as fields. This is useful for representing hierarchical data, such as JSON objects with nested structures. By using nested models, you can ensure that each part of the data adheres to its own validation rules, making your data handling more robust and maintainable.

In [35]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str=Field(description="Title of the movie")
    year:int=Field(description="year of the release")
    cast:list[Actor]
    genres:list[Actor]
    buget:float|None=Field(None,description="Budget in million USD")

In [36]:
model_so=model.with_structured_output(MovieDetails)

In [37]:
model_so.invoke("Bahubale")

MovieDetails(title='Bahubali', year=2015, cast=[Actor(name='Prabhas', role='Mahendra Baahubali'), Actor(name='Rana Daggubati', role='Kattappa'), Actor(name='Anushka Shetty', role='Devasena')], genres=[Actor(name='Action', role='Action'), Actor(name='Drama', role='Drama')], buget=None)

# TypeDict

In [40]:
from typing_extensions import TypedDict, Annotated
class MovieDict(TypedDict):
    """A movie with details"""
    title:Annotated[str,...,"The title of the movie"]
    year:Annotated[int,...,"The year the movie was released"]
    director:Annotated[str,...,"The director of the movie"]
    rating:Annotated[float,...,"The movie's rating out of 10"]

model_so=model.with_structured_output(MovieDict)
model_so.invoke("ddlj")

{'director': 'Aditya Chopra',
 'rating': 8.5,
 'title': 'Dilwale Dulhania Le Jayenge',
 'year': 1995}

# DataClasses

In [48]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain_groq import ChatGroq

@dataclass
class ContactInfo:
    name: str
    email: str
    phone: str


model = ChatGroq(
    model="openai/gpt-oss-20b"
)

agent = create_agent(
    model=model,
    response_format=ContactInfo
)

In [51]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "My name is Rajib. My email is rajib@example.com and my phone is 9876543210."
        }
    ]
})

result['structured_response']

ContactInfo(name='Rajib', email='rajib@example.com', phone='9876543210')